In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("dtl_data.csv")
df

,NOSE_X,NOSE_Y,NOSE_Z,NOSE_V,LEFT_EYE_INNER_X,LEFT_EYE_INNER_Y,LEFT_EYE_INNER_Z,LEFT_EYE_INNER_V,LEFT_EYE_X,LEFT_EYE_Y,...,LEFT_FOOT_INDEX_X,LEFT_FOOT_INDEX_Y,LEFT_FOOT_INDEX_Z,LEFT_FOOT_INDEX_V,RIGHT_FOOT_INDEX_X,RIGHT_FOOT_INDEX_Y,RIGHT_FOOT_INDEX_Z,RIGHT_FOOT_INDEX_V,Pose_Class,Image_Path
0,0.206633,-0.455494,-0.302457,0.998422,0.210564,-0.484885,-0.282526,0.997703,0.210146,-0.487218,...,0.148562,0.730507,0.464938,0.920501,0.067307,0.835685,-0.621484,0.994325,P1,P1/eric_DTL_1_frame0.jpg
1,0.208898,-0.453553,-0.252352,0.998478,0.212801,-0.485189,-0.230236,0.997794,0.212357,-0.487674,...,0.147901,0.729127,0.491692,0.921419,0.068346,0.831762,-0.616651,0.994507,P1,P1/eric_DTL_1_frame1.jpg
2,0.210588,-0.452270,-0.275182,0.998537,0.214489,-0.485385,-0.250130,0.997884,0.214039,-0.487981,...,0.149697,0.728278,0.484572,0.921430,0.069607,0.831347,-0.608091,0.994595,P1,P1/eric_DTL_1_frame2.jpg
3,0.210899,-0.451608,-0.276378,0.998601,0.214706,-0.485893,-0.253785,0.997988,0.214208,-0.488479,...,0.147934,0.726683,0.469122,0.921157,0.069664,0.828168,-0.652296,0.994714,P1,P1/eric_DTL_1_frame3.jpg
4,0.212667,-0.454765,-0.299136,0.998663,0.216285,-0.489414,-0.276320,0.998082,0.215762,-0.491862,...,0.149706,0.728206,0.411359,0.920982,0.070610,0.829205,-0.718786,0.994804,P1,P1/eric_DTL_1_frame4.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384,0.178441,-0.384474,0.435407,0.996624,0.183764,-0.401237,0.403489,0.996592,0.181154,-0.408924,...,0.051336,0.699496,-0.075233,0.625570,0.024945,0.760798,-0.204813,0.946430,P9,P9/tom_DTL_1_frame56.jpg
385,0.174149,-0.395745,0.453819,0.996951,0.178817,-0.412013,0.420745,0.996925,0.175792,-0.419947,...,0.019983,0.529721,0.344876,0.603410,0.024494,0.752194,-0.425548,0.947251,P9,P9/tom_DTL_1_frame57.jpg
386,0.177846,-0.430155,0.116032,0.996852,0.179398,-0.463835,0.089544,0.996878,0.174692,-0.475034,...,0.061596,0.640283,0.412371,0.608406,0.084231,0.714363,-0.020284,0.949056,P9,P9/tom_DTL_2_frame43.jpg
387,0.177185,-0.440576,0.161013,0.997003,0.182096,-0.457049,0.124483,0.997010,0.177286,-0.468118,...,0.050690,0.641206,0.233254,0.630993,0.081213,0.722277,-0.067027,0.950351,P9,P9/tom_DTL_2_frame44.jpg


In [15]:
import re

def extract_group(path):
    match = re.search(r'P\d+/(.*?)_', path) # Person's name
    # match = re.search(r'P\d+/(.*)_frame\d+', path) # Group by video clips
    return match.group(1) if match else None

df['group'] = df['Image_Path'].apply(extract_group)
group_counts = df['group'].value_counts()
display(group_counts)

# Display unique classes per group
group_classes = df.groupby('group')['Pose_Class'].unique()
display(group_classes)

# Show pose counts per person (group)
pose_per_person = df.groupby(['group', 'Pose_Class']).size().unstack(fill_value=0)
display(pose_per_person)

group
rob       139
grant      82
tom        81
eric       46
random     41
Name: count, dtype: int64

group
eric      [P1, P10, P2, P3, P4, P5, P6, P7, P8, P9]
grant     [P1, P10, P2, P3, P4, P5, P6, P7, P8, P9]
random    [P1, P10, P2, P3, P4, P5, P6, P7, P8, P9]
rob       [P1, P10, P2, P3, P4, P5, P6, P7, P8, P9]
tom       [P1, P10, P2, P3, P4, P5, P6, P7, P8, P9]
Name: Pose_Class, dtype: object

Pose_Class,P1,P10,P2,P3,P4,P5,P6,P7,P8,P9
group,,,,,,,,,,
eric,7,7,7,7,7,4,1,2,1,3
grant,9,9,8,8,6,7,8,10,8,9
random,7,4,8,4,6,3,1,3,2,3
rob,18,15,14,18,15,13,7,14,13,12
tom,13,9,13,14,11,5,2,4,2,8


In [16]:
import pandas as pd

# 1. Identify the 5 full-class persons
pose_per_person = df.groupby(['group', 'Pose_Class']).size().unstack(fill_value=0)

# Find persons who have all 10 classes (non-zero count for all)
full_class_persons = pose_per_person[(pose_per_person > 0).all(axis=1)].index.tolist()

print("Full-class persons:", full_class_persons)

# 2. Assign each full-class person to a unique big group
num_big_groups = 5
big_group_labels = [f'BigGroup_{i+1}' for i in range(num_big_groups)]

assert len(full_class_persons) == num_big_groups, "Number of full-class persons must equal number of big groups"

assignment = {}
for i, person in enumerate(full_class_persons):
    assignment[person] = big_group_labels[i]

# 3. Assign the remaining persons greedily to balance classes
remaining_persons = [p for p in pose_per_person.index if p not in full_class_persons]

# Initialize big group pose sums with full-class persons included
big_group_sums = pd.DataFrame(0, index=big_group_labels, columns=pose_per_person.columns)
for person, big_group in assignment.items():
    big_group_sums.loc[big_group] += pose_per_person.loc[person]

# Greedy assign remaining persons
for person in remaining_persons:
    best_group = None
    best_balance_score = None
    for group in big_group_labels:
        new_totals = big_group_sums.loc[group] + pose_per_person.loc[person]
        imbalance = new_totals.std()
        if best_balance_score is None or imbalance < best_balance_score:
            best_balance_score = imbalance
            best_group = group
    assignment[person] = best_group
    big_group_sums.loc[best_group] += pose_per_person.loc[person]

# 4. Map assignment back to original dataframe
df['big_group'] = df['group'].map(assignment)

# 5. Check final distribution
big_group_pose_counts = df.groupby(['big_group', 'Pose_Class']).size().unstack(fill_value=0)
display(big_group_pose_counts)

Full-class persons: ['eric', 'grant', 'random', 'rob', 'tom']


Pose_Class,P1,P10,P2,P3,P4,P5,P6,P7,P8,P9
big_group,,,,,,,,,,
BigGroup_1,7,7,7,7,7,4,1,2,1,3
BigGroup_2,9,9,8,8,6,7,8,10,8,9
BigGroup_3,7,4,8,4,6,3,1,3,2,3
BigGroup_4,18,15,14,18,15,13,7,14,13,12
BigGroup_5,13,9,13,14,11,5,2,4,2,8


In [ ]:
# print or use the mapping dictionary:
print("Person to Big Group mapping:")
for person, big_group in assignment.items():
    print(f"{person} -> {big_group}")

# Create a DataFrame
mapping_df = pd.DataFrame(list(assignment.items()), columns=['person', 'big_group'])
display(mapping_df)

# Save to CSV
mapping_df.to_csv('person_to_big_group_mapping.csv', index=False)

Person to Big Group mapping:
eric -> BigGroup_1
grant -> BigGroup_2
random -> BigGroup_3
rob -> BigGroup_4
tom -> BigGroup_5


,person,big_group
0,eric,BigGroup_1
1,grant,BigGroup_2
2,random,BigGroup_3
3,rob,BigGroup_4
4,tom,BigGroup_5


In [18]:
assignment_int = {person: int(group.split('_')[1]) for person, group in assignment.items()}
assignment_int

{'eric': 1, 'grant': 2, 'random': 3, 'rob': 4, 'tom': 5}

In [ ]:
import pandas as pd
import re

df4 = pd.read_csv('dtl_data.csv')

# Extract person name from Image_Path
def extract_person(path):
    match = re.search(r'P\d+/(.*?)_', path)
    return match.group(1) if match else None

# Extract person name for each row
df4['person'] = df4['Image_Path'].apply(extract_person)

# Map persons to big groups, fill missing with 'Unknown'
df4['group'] = df4['person'].map(assignment_int).fillna('Unknown')

# Drop helper column if you want
df4.drop(columns=['person'], inplace=True)

# Save the updated dataframe
df4.to_csv('dtl_data_groupbynames.csv', index=False)

print("Done! 'big_group' column added and saved to dtl_data_groupbynames.csv")

Done! 'big_group' column added and saved to dataset4_with_big_groups.csv
